# Tessera — Portable Validation (Google Colab)

This notebook checks the portable Windows-mock tests using GCC, Clang, and GCC ASan/UBSan.

**Limitations:** This does not validate real Windows inline hooking, x64 unwind registration, MSVC, Detours, or DLL loader behavior.

## 0. Private GitHub access

In Colab, open **Secrets** (key icon), add a secret named `GH_TOKEN`, paste your fine-grained token, and enable notebook access. Grant read-only **Contents** permission for `Jue-dev/Tessera`. Do not paste the token into notebook cells.


In [ ]:
from google.colab import userdata
import os, subprocess, pathlib, time, shutil, json

TOKEN = userdata.get("GH_TOKEN")
if not TOKEN:
    raise RuntimeError("GH_TOKEN missing. Add it in Colab Secrets and enable notebook access.")

REPO = "https://github.com/Jue-dev/Tessera.git"
BRANCH = "issue-6/unwind-lifecycle"
WORKDIR = pathlib.Path("/content/tessera")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

askpass = pathlib.Path("/content/tessera_git_askpass.sh")
askpass.write_text("\n".join([
    "#!/bin/sh",
    'case "$1" in *Username*) echo x-access-token ;; *) echo "$GIT_PASSWORD" ;; esac',
    ""
]))
askpass.chmod(0o700)

env = os.environ.copy()
env.update({
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_PASSWORD": TOKEN
})
try:
    clone = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORKDIR)],
        env=env, text=True, capture_output=True
    )
finally:
    askpass.unlink(missing_ok=True)
    env.pop("GIT_PASSWORD", None)
    TOKEN = None

if clone.returncode != 0:
    print(clone.stderr)
    raise RuntimeError(f"Clone failed (exit {clone.returncode}). Check token permissions, repo and branch.")

os.chdir(WORKDIR)
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Clone successful")
print("Branch:", BRANCH)
print("HEAD:", HEAD)



## Test helper

Each test runs separately, saves its log, and records PASS/FAIL. The summary cell at the end raises an error if any validation fails.


In [ ]:
LOG_DIR = pathlib.Path("/content/tessera-colab-logs")
LOG_DIR.mkdir(exist_ok=True)
results = []

def run_test(name, command, extra_env=None):
    print("\\n===== " + name + " =====")
    run_env = os.environ.copy()
    if extra_env:
        run_env.update(extra_env)
    start = time.time()
    proc = subprocess.run(command, cwd=WORKDIR, env=run_env, text=True, capture_output=True)
    elapsed = round(time.time() - start, 2)
    log = "\\n".join([
        "$ " + " ".join(command),
        "",
        "--- stdout ---",
        proc.stdout,
        "--- stderr ---",
        proc.stderr,
        "EXIT=" + str(proc.returncode)
    ])
    log_path = LOG_DIR / (name.lower().replace(" ", "_") + ".log")
    log_path.write_text(log)
    print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    status = "PASS" if proc.returncode == 0 else "FAIL"
    results.append({"test": name, "status": status, "exit_code": proc.returncode, "seconds": elapsed})
    print(f"{status} (exit {proc.returncode}; {elapsed}s)")
    return proc.returncode



## 1. GCC mock backend tests


In [ ]:
run_test("GCC mock", ["bash", "verify/windows_mock/run_windows_mock_tests.sh"])


## 2. Clang mock backend tests

Clang will be installed if it is not present in the Colab runtime.


In [ ]:
if shutil.which("clang++") is None:
    install = subprocess.run(
        ["bash", "-lc", "apt-get update -qq && apt-get install -y clang"],
        text=True, capture_output=True
    )
    print(install.stdout)
    if install.returncode:
        print(install.stderr)

if shutil.which("clang++"):
    run_test("Clang mock", ["bash", "verify/windows_mock/run_windows_mock_tests.sh"], {"CXX": "clang++"})
else:
    results.append({"test": "Clang mock", "status": "FAIL", "exit_code": 127, "seconds": None})
    print("Clang unavailable; marked FAIL.")



## 3. GCC AddressSanitizer + UndefinedBehaviorSanitizer


In [ ]:
run_test(
    "GCC ASan UBSan",
    ["bash", "verify/windows_mock/run_windows_mock_tests.sh"],
    {
        "EXTRA_CXXFLAGS": "-fsanitize=address,undefined -fno-omit-frame-pointer",
        "ASAN_OPTIONS": "detect_leaks=1:halt_on_error=1",
        "UBSAN_OPTIONS": "halt_on_error=1:print_stacktrace=1"
    }
)


## 4. Summary and logs


In [ ]:
print("Validated branch:", BRANCH)
print("Validated HEAD:", HEAD)
print(json.dumps(results, indent=2))
summary = {"branch": BRANCH, "head": HEAD, "results": results}
(LOG_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
subprocess.run(["git", "status", "--short"], check=False)

failures = [r for r in results if r["status"] != "PASS"]
if failures:
    raise RuntimeError(f"{len(failures)} validation(s) failed. Logs: {LOG_DIR}")
print("ALL PORTABLE VALIDATIONS PASSED")

